In [1]:
import os,re
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_core.output_parsers import CommaSeparatedListOutputParser,StrOutputParser

from src.ingestion import load_pdf
from src.embeddings import get_embedding_model, store_embeddings
from src.chunkers import fixed_chunker, header_chunker, parent_child_chunker
from src.retriever import is_multi_topic,extract_sections,get_relevant_sections 
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

/Users/itsmesuryaa/Python/Kaggle_Dataset/GenAI_Krish/genai_project/lggen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**run_agent()**  → master orchestrator (decides who runs next). 
**state**       → shared memory (stores everyone's results). 
**nodes**        → specialists (do the actual work). 

In [2]:
#pdf_path = "langchain_rag_technical_docs_clean.pdf"

# pages = load_pdf(pdf_path)
# embedding = get_embedding_model()
# header_chunks = header_chunker.chunk(pages)
# header_embedding = store_embeddings(header_chunks, "Header_Chunks", embedding)
# llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0)

In [3]:
def build_pipeline(pdf_path):
    print("Loading PDF...")
    pages = load_pdf(pdf_path)

    print("Loading embedding model...")
    embedding = get_embedding_model()

    print("Chunking...")
    fixed_chunks  = fixed_chunker.chunk(pages)
    header_chunks = header_chunker.chunk(pages)
    child_chunks  = parent_child_chunker.create_child_chunks(header_chunks)
    parent_chunks = parent_child_chunker.create_parent_chunks(child_chunks)

    print(f"Fixed chunks  : {len(fixed_chunks)}")
    print(f"Header chunks : {len(header_chunks)}")
    print(f"Child chunks  : {len(child_chunks)}")

    print("Storing embeddings...")
    fixed_embedding  = store_embeddings(fixed_chunks,  "Fixed_Chunks",  embedding)
    header_embedding = store_embeddings(header_chunks, "Header_Chunks", embedding)
    child_embedding  = store_embeddings(child_chunks,  "child_chunks",  embedding)

    print("Storing parent chunks...")
    parent_child_chunker.store_parent_chunks(parent_chunks)

    return (
        embedding,
        fixed_embedding,  fixed_chunks,
        header_embedding, header_chunks,
        child_embedding,  child_chunks
    )


In [4]:
## embedding, _, _, header_embedding, header_chunks, *_ = build_pipeline("langchain_rag_technical_docs_clean.pdf")


In [5]:
## Agent/State.py

## State -- it store each AI agent results
# state_history= {} 
def create_state(query,header_chunks,embedding,header_embedding,child_embedding,fixed_embedding,fixed_chunks,child_chunks):
    state = { "query":       query,  # comes from user
        "chunks":      [],     # filled later by retrieve_node
        "grade":       "",     # filled later by grade_node
        "answer":      "",     # filled later by answer_node
        "retry_count": 0,      # starts at zero
        "max_retry":   4,      # fixed limit
        "header_chunks": header_chunks,
        "header_embedding": header_embedding,
        "child_embedding": child_embedding,
        "fixed_embedding": fixed_embedding,
        "fixed_chunks": fixed_chunks,
        "child_chunks": child_chunks,
        "embedding" : embedding,
        "current_strategy": "header",
        "sections": [],
        "sub_questions": [],
        "direct_query":""
    }
    return state


In [6]:
## Node.py

## Classifier node :

def classifier_node(state) -> dict:
    query = state['query']
    result = is_multi_topic(query)
    state['classifier'] = result
    return state

## Section node.py

## Section Node
## Why: LLM needs to know WHICH sections exist in the document
## before generating sub-questions. Without this, LLM generates
## broad random questions → retrieves irrelevant chunks.
## We use header_chunks because header strategy preserves 
## section headings (Header 1, Header 2) in metadata.
## We do NOT hardcode header_chunks inside this node because
## chunking already happened in build_pipeline() — 
## we just read it from state (lunchbox pattern).

def section_node(state)->dict:

    query         = state["query"]
    header_chunks = state["header_chunks"]
    embedding     = state["embedding"]
    
    sections = get_relevant_sections(query, header_chunks, embedding)
    
    state["sections"] = sections
    return state

In [7]:
embedding, fixed_embedding, fixed_chunks, header_embedding, header_chunks, child_embedding, child_chunks = build_pipeline("langchain_rag_technical_docs_clean.pdf")


Loading PDF...
Loading embedding model...
Chunking...
Fixed chunks  : 151
Header chunks : 59
Child chunks  : 59
Storing embeddings...
Storing parent chunks...


In [8]:
##state = create_state(query = "what is parent-child chunking and explain the code as well?",header_chunks=header_chunks,embedding=embedding,header_embedding=header_embedding,child_embedding=child_embedding,fixed_embedding=fixed_embedding,fixed_chunks=fixed_chunks,child_chunks=child_chunks) 

Explain the different types of retrieval strategies

In [9]:
state = create_state(query = "Explain about different types of retrieval strategies like HyDE, Query Decomposition and Self -Correction?",
                     header_chunks=header_chunks,embedding=embedding,header_embedding=header_embedding,child_embedding=child_embedding,fixed_embedding=fixed_embedding,fixed_chunks=fixed_chunks,child_chunks=child_chunks) 

In [10]:
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0)

In [11]:
clas_st = classifier_node(state)

In [12]:
sec_st = section_node(state)

In [13]:
state['sub_questions']

[]

In [14]:
state['query']

'Explain about different types of retrieval strategies like HyDE, Query Decomposition and Self -Correction?'

In [15]:
sec_st['sections']

['langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.3 Reranking Retrieved Results-25',
 'langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.1 Similarity Search vs MMR-23',
 'langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.2 Hybrid Search (Keyword + Semantic)-24',
 'langchain_rag_technical_docs_clean.pdf-Chapter 1: Introduction to RAG-1.1 What is Retrieval-Augmented Generation?-0',
 'langchain_rag_technical_docs_clean.pdf-Chapter 11: Advanced RAG Patterns-11.2 Query Decomposition-33',
 'langchain_rag_technical_docs_clean.pdf-Chapter 6: Vector Stores-6.1 ChromaDB Setup and Usage-20',
 'langchain_rag_technical_docs_clean.pdf-Appendix C: ChromaDB Advanced Usage-C.3 Updating and Deleting Chunks-49',
 'langchain_rag_technical_docs_clean.pdf-Appendix F: RAG Interview Questions & Model Answers-F.3 Debugging Scenarios-58']

In [16]:
## Decompose node

# decompose_prompt = PromptTemplate(
#     input_variables=["question", "sections"],
#     template="""
# You are an expert search query planner for a RAG system.

# Available sections in the document:
# {sections}

# User question: {question}

# Step 1 — From the available sections, identify ONLY the sections directly relevant to the user question. Ignore all unrelated sections.
# Step 2 — Generate 3-5 specific sub-queries, each targeting one of the relevant sections identified in Step 1.

# Rules:
# - Only use sections that are clearly relevant to the question
# - Do NOT generate sub-queries for unrelated sections
# - Do NOT invent topics not covered by the relevant sections
# - Return ONLY a comma-separated list of sub-queries

# Sub-questions:"""
# )

# def decompose_node(state):
#      parser = CommaSeparatedListOutputParser()
#      question = state["query"]
#      sections = state['sections']

#      chain = decompose_prompt | llm | parser
#      sub_questions = chain.invoke({
#             "question": question,
#             "sections": "\n".join(sections)
#         })
#      state["sub_questions"] = sub_questions
#      return state 


In [17]:
decompose_prompt = PromptTemplate(
    input_variables=["question", "sections"],
    template="""
You are an expert search query planner for a RAG system.

Available sections in the document:
{sections}

User question: {question}

Step 1 — For each section above, decide: is this section topically relevant to the user question?
         A section is relevant ONLY if its title directly addresses a concept asked in the question.
         Discard any section whose title does not relate to the question.

Step 2 — From the relevant sections identified in Step 1, generate 3-5 specific sub-queries.
         Each sub-query must be directly answerable from one of those relevant sections.

Rules:
- Sub-queries must be grounded in the user question, not invented from section titles alone
- Do NOT generate a sub-query for a section that is irrelevant to the user question
- Do NOT invent topics not present in the user question
- Return ONLY a comma-separated list of sub-queries

Sub-questions:"""
)

def decompose_node(state):
     parser = CommaSeparatedListOutputParser()
     question = state["query"]
     sections = state['sections']

     chain = decompose_prompt | llm | parser
     sub_questions = chain.invoke({
            "question": question,
            "sections": "\n".join(sections)
        })
     state["sub_questions"] = sub_questions
     return state

In [18]:
rewrite_prompt = PromptTemplate(
            input_variables=["question", "sections"],
            template="""
        You are a search query rewriter for a RAG system about LangChain documentation.
        Rules:
      - Rewrite the question using ONLY topics found in the sections above
      - Do NOT add topics outside the sections
      - One line only, no prefix

        Available sections: {sections}
        Question: {question}
        Rewritten query (one line only):"""
        )

def single_rewriter_node(state):
        
        question = state["query"]
        sections = state['sections']
        
        
        chain = rewrite_prompt | llm | StrOutputParser()
        rewritten = chain.invoke({
            "question": question,
            "sections": "\n".join(sections)
        })
        state['direct_query'] = rewritten
        return state

In [19]:
def query_planner_node(state) -> dict:
    if state['classifier']:
        state = decompose_node(state)
       
    else:
        state = single_rewriter_node(state)
    return state

In [20]:
cn = query_planner_node(state)

In [21]:
cn['sections']

['langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.3 Reranking Retrieved Results-25',
 'langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.1 Similarity Search vs MMR-23',
 'langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.2 Hybrid Search (Keyword + Semantic)-24',
 'langchain_rag_technical_docs_clean.pdf-Chapter 1: Introduction to RAG-1.1 What is Retrieval-Augmented Generation?-0',
 'langchain_rag_technical_docs_clean.pdf-Chapter 11: Advanced RAG Patterns-11.2 Query Decomposition-33',
 'langchain_rag_technical_docs_clean.pdf-Chapter 6: Vector Stores-6.1 ChromaDB Setup and Usage-20',
 'langchain_rag_technical_docs_clean.pdf-Appendix C: ChromaDB Advanced Usage-C.3 Updating and Deleting Chunks-49',
 'langchain_rag_technical_docs_clean.pdf-Appendix F: RAG Interview Questions & Model Answers-F.3 Debugging Scenarios-58']

In [22]:
def retriever_node(state) -> dict:
    strategy = state['current_strategy']
    
    if strategy == "header":
        embedding_store = state['header_embedding']
        chunk_store     = state['header_chunks']
    elif strategy == "child":
        embedding_store = state['child_embedding']
        chunk_store     = state['child_chunks']
    else:
        embedding_store = state['fixed_embedding']
        chunk_store     = state['fixed_chunks']
    
    queries = state['sub_questions'] or [state['direct_query']]
    
    seen, all_chunks = set(), []
    for q in queries:
        bm25 = BM25Retriever.from_documents(chunk_store)
        bm25.k = 3
        vector = embedding_store.as_retriever(search_kwargs={"k": 3})
        hybrid = EnsembleRetriever(retrievers=[bm25, vector], weights=[0.4, 0.6])
        for doc in hybrid.invoke(q):
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                all_chunks.append(doc)
    
    state['chunks'] = all_chunks
    return state

In [23]:
ret_no = retriever_node(state)

In [24]:
state['chunks']

[Document(id='b23a47bccb62ab383357d6fbe2e09052', metadata={'Chunk_id': 'header_23', 'Header 2': '7.1 Similarity Search vs MMR', 'strategy': 'header', 'Header 1': 'Chapter 7: Retrieval Strategies', 'header_id': 'uploaded_doc.pdf-Chapter 7: Retrieval Strategies-7.1 Similarity Search vs MMR-23', 'total_chunks': 59, 'hash': 'b23a47bccb62ab383357d6fbe2e09052', 'source': 'uploaded_doc.pdf'}, page_content='7.1 Similarity Search vs MMR-Two main retrieval modes: Similarity Search returns the top-k most similar chunks. Maximum Marginal\nRelevance (MMR) balances relevance with diversity, avoiding redundant results.\n# Standard similarity search - may return redundant chunks\nresults_similarity = vectorstore.similarity_search(\nquery="password reset",\nk=5\n)\n# MMR - balances relevance AND diversity\nresults_mmr = vectorstore.max_marginal_relevance_search(\nquery="password reset",\nk=5,\nfetch_k=20,     # fetch 20, then select 5 diverse ones\nlambda_mult=0.7 # 0=max diversity, 1=max relevance\n)'

In [25]:
## Single doc metadata
# state['chunks'][0].metadata

# First 3 docs metadata
[doc.metadata.get('Header 2') for doc in state['chunks']]

['7.1 Similarity Search vs MMR',
 '11.3 Self-RAG — Retrieval on Demand',
 'F.1 Conceptual Questions',
 '5.1 What Are Embeddings?',
 'E.3 Golden Dataset for Evaluation',
 '8.2 Multi-Strategy Chain Comparison',
 '6.2 Metadata Filtering (Key Differentiator)',
 'F.2 System Design Questions',
 '12.1 Streamlit App Structure',
 '11.1 HyDE — Hypothetical Document Embeddings',
 None]

In [26]:
from src.reranker import rerank 

## Reranker node
## if parent-child → top 3, if multi-topic → top 5, else top 2 (we want more docs for multi-topic to increase recall, and less docs for single-topic to increase precision)
## top_n is a hyperparameter that can be tuned based on the use case and observed performance. The rationale is that multi-topic queries benefit from a broader set of retrieved chunks to cover all aspects, while single-topic queries require a more focused set to enhance precision.
import time 

# def reranker_node(state) -> dict:
#     query = state['query']
#     chunks = state['chunks']

#     if state["current_strategy"] == "child":
#         top_n = 3
#     elif state["classifier"]:      # multi-topic
#         top_n = 5
#     else:                          # single/direct query
#         top_n = 2 

#     try:
#         time.sleep(10)
#         reranked        = rerank(query, chunks, top_n=top_n)
#         state['reranked_chunks'] = reranked
#     except Exception as e:
#         print(f"Reranker failed: {e}")
#         print("Skipping reranker — using raw chunks")
#         # chunks stay as is, no scores added
    
#     return state




In [41]:
from src.reranker import rerank 
def reranker_node(state) -> dict:
    chunks = state['chunks']

    if state["classifier"]:
        # Multi-topic: rerank each sub-question separately
        seen, reranked_all = set(), []
        for sub_q in state['sub_questions']:
            time.sleep(10)
            reranked = rerank(sub_q, chunks, top_n=3)
            for doc in reranked:
                if doc.page_content not in seen:
                    seen.add(doc.page_content)
                    reranked_all.append(doc)
        state['reranked_chunks'] = reranked_all
    else:
        # Single query: rerank against direct_query (already rewritten)
        top_n = 3 if state["current_strategy"] == "child" else 2
        state['reranked_chunks'] = rerank(state['direct_query'], chunks, top_n=top_n)

    return state


In [28]:
re_nose = reranker_node(state)

In [29]:
for doc in state['reranked_chunks']:
    print(doc.metadata.get('Header 2'))
    

8.2 Multi-Strategy Chain Comparison
7.1 Similarity Search vs MMR
F.1 Conceptual Questions
11.3 Self-RAG — Retrieval on Demand
6.2 Metadata Filtering (Key Differentiator)
11.1 HyDE — Hypothetical Document Embeddings
None


In [30]:
state['sub_questions']

['What are the different types of retrieval strategies',
 'How does Query Decomposition work as a retrieval strategy',
 'What is HyDE retrieval strategy',
 'How does Self-Correction work in retrieval strategies']

In [31]:
## Grade node
## instead of fiexd threshold, we use dynamic threshold based on average relevance score of retrieved chunks. This allows flexibility across different queries and document sets, as the threshold adapts to the quality of retrieved chunks. If no chunks meet or exceed the average score, we trigger a retry with a different strategy to improve retrieval results.
## we also added "exhausted" state when retry_count exceeds max_retry, to prevent infinite retries and provide user feedback.


STRATEGIES = ["header", "child", "fixed"]

def grade_node(state) -> dict:
    chunks = state['reranked_chunks']

    if state['retry_count'] >= state['max_retry']:
             state['grade']  = "exhausted"
             state['answer'] = "I could not find relevant information for your query. Please try rephrasing your question."
             return state
 
    scores    = [doc.metadata.get('relevance_score', 0) for doc in chunks]
    print("Relevance scores of retrieved chunks:", scores)
    avg_score = sum(scores) / len(scores) if scores else 0
    print("Average relevance score:", avg_score) 


    MIN_QUALITY_THRESHOLD = 0.30

    if avg_score < MIN_QUALITY_THRESHOLD:
         state['grade']            = "fail"
         next_retry                = state['retry_count'] + 1
         state['retry_count']      = next_retry
         state['current_strategy'] = STRATEGIES[next_retry % len(STRATEGIES)]
         return state
    
    good_chunks = [doc for doc in chunks 
                   if doc.metadata.get('relevance_score', 0) >= avg_score]
    
    if good_chunks:
        state['grade']  = "pass"
        state['reranked_chunks'] = good_chunks
#     else:
         
#          state['grade']            = "fail"
#          next_retry                = state['retry_count'] + 1
#          state['retry_count']      = next_retry
#          state['current_strategy'] = STRATEGIES[next_retry % len(STRATEGIES)]
    
    return state


In [32]:
grade = grade_node(state)
print(grade)


Relevance scores of retrieved chunks: [0.3083985, 0.02470387, 0.0054905633, 0.003324437, 0.00017130819, 0.57660156, 9.972941e-06]
Average relevance score: 0.1312428873472857
{'query': 'Explain about different types of retrieval strategies like HyDE, Query Decomposition and Self -Correction?', 'chunks': [Document(id='b23a47bccb62ab383357d6fbe2e09052', metadata={'Chunk_id': 'header_23', 'Header 2': '7.1 Similarity Search vs MMR', 'strategy': 'header', 'Header 1': 'Chapter 7: Retrieval Strategies', 'header_id': 'uploaded_doc.pdf-Chapter 7: Retrieval Strategies-7.1 Similarity Search vs MMR-23', 'total_chunks': 59, 'hash': 'b23a47bccb62ab383357d6fbe2e09052', 'source': 'uploaded_doc.pdf'}, page_content='7.1 Similarity Search vs MMR-Two main retrieval modes: Similarity Search returns the top-k most similar chunks. Maximum Marginal\nRelevance (MMR) balances relevance with diversity, avoiding redundant results.\n# Standard similarity search - may return redundant chunks\nresults_similarity = ve

In [33]:
for doc in state['chunks']:
    print(doc.metadata)
    break

{'Chunk_id': 'header_23', 'Header 2': '7.1 Similarity Search vs MMR', 'strategy': 'header', 'Header 1': 'Chapter 7: Retrieval Strategies', 'header_id': 'uploaded_doc.pdf-Chapter 7: Retrieval Strategies-7.1 Similarity Search vs MMR-23', 'total_chunks': 59, 'hash': 'b23a47bccb62ab383357d6fbe2e09052', 'source': 'uploaded_doc.pdf'}


In [34]:
## Answer node
## context -> chunks comtain the relevant information to answer the question. We concatenate the content of retrieved chunks to create a context passage for the LLM. The prompt instructs the LLM to answer the question using ONLY the provided context, which helps to ensure that the answer is grounded in the retrieved information and reduces hallucination.


def answer_node(state) -> dict:
    query   = state["query"]
    chunks  = state["reranked_chunks"]
    context = "\n\n".join([doc.page_content for doc in chunks]) 
    #print("context passed to LLM:",context)
    prompt = PromptTemplate(
        input_variables=["query", "context"],
        template="""Answer the question using only the context below.
        Rules:
- Include complete code examples exactly as they appear in context
- Do not summarize or shorten code blocks
- Structure your answer clearly

Context:
{context}

Question: {query}
Answer:"""
    )

    chain          = prompt | llm | StrOutputParser()
    state["answer"] = chain.invoke({"query": query, "context": context})
    return state


In [35]:
an = answer_node(state)
print(an['answer'])

### Introduction to Retrieval Strategies

Retrieval strategies are crucial components of a Retrieval-Augmented Generation (RAG) pipeline, as they determine how relevant information is retrieved from a knowledge base to answer a given question. This section will discuss different types of retrieval strategies, including HyDE, Query Decomposition, and Self-Correction.

### 1. HyDE (Hypothetical Document Embeddings)

HyDE is a retrieval strategy that involves generating a hypothetical answer to a question before embedding it. This approach is based on the idea that a hypothetical answer is semantically closer to the real answer chunks than the original question.

```python
from langchain.chains import HypotheticalDocumentEmbedder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-3.5-turbo")
base_embeddings = OpenAIEmbeddings()

hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=N

In [36]:
state['query']

'Explain about different types of retrieval strategies like HyDE, Query Decomposition and Self -Correction?'

In [37]:
state['sub_questions']

['What are the different types of retrieval strategies',
 'How does Query Decomposition work as a retrieval strategy',
 'What is HyDE retrieval strategy',
 'How does Self-Correction work in retrieval strategies']

In [40]:
state['sections']

['langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.3 Reranking Retrieved Results-25',
 'langchain_rag_technical_docs_clean.pdf-Chapter 7: Retrieval Strategies-7.1 Similarity Search vs MMR-23',
 'langchain_rag_technical_docs_clean.pdf-Chapter 11: Advanced RAG Patterns-11.2 Query Decomposition-33',
 'langchain_rag_technical_docs_clean.pdf-Chapter 6: Vector Stores-6.1 ChromaDB Setup and Usage-20',
 'langchain_rag_technical_docs_clean.pdf-Appendix C: ChromaDB Advanced Usage-C.3 Updating and Deleting Chunks-49',
 'langchain_rag_technical_docs_clean.pdf-Appendix F: RAG Interview Questions & Model Answers-F.3 Debugging Scenarios-58']

In [38]:
## run_agent.py

def run_agent(state):
    
    # Step 1 - Find relevant sections
    state = section_node(state)
    
    # Step 2 - Classify and plan query
    state = classifier_node(state)
    state = query_planner_node(state)
    
    # Step 3 - Retry loop
    while True:
    #state['retry_count'] <= state['max_retry']:
        
        state = retriever_node(state)
        state = reranker_node(state)
        state = grade_node(state)
        
        if state['grade'] in ("pass",'exhausted'):
            break
        # elif state['grade'] == "fail":
        #     continue   # retry with new strategy
    
    # Step 4 - Answer
    if state['grade'] == "pass":
        
        state = answer_node(state)
    
    return state

In [42]:
# Run agent
state = run_agent(state)

# # Results
# print(f"Grade:         {state['grade']}")
# print(f"Retry count:   {state['retry_count']}")
# print(f"Strategy used: {state['current_strategy']}")
print(f"\nAnswer:\n{state['answer']}")

Relevance scores of retrieved chunks: [0.9672869, 0.0021322616, 0.0020348171, 0.98866826, 9.387641e-05]
Average relevance score: 0.392043223022

Answer:
## Retrieval Strategies
There are several retrieval strategies that can be employed to improve the efficiency and effectiveness of information retrieval. This answer will focus on three types of retrieval strategies: Similarity Search, HyDE, and Query Decomposition.

### Similarity Search
Similarity Search is a retrieval mode that returns the top-k most similar chunks based on a given query. However, this approach may return redundant chunks.

```python
# Standard similarity search - may return redundant chunks
results_similarity = vectorstore.similarity_search(
    query="password reset",
    k=5
)
```

### HyDE
HyDE (Hybrid Dense Embeddings) is a type of embedding that can be used as a drop-in replacement for other embeddings. It can be used with a vector store to perform similarity searches.

```python
# Use HyDE embeddings as drop-

In [ ]:
state['chunks']

[Document(id='9debbf556a3c412aca436dd65cd56def', metadata={'hash': '9debbf556a3c412aca436dd65cd56def', 'source': 'uploaded_doc.pdf', 'header_id': 'uploaded_doc.pdf-Chapter 4: Chunking Strategies-MISSING-11', 'total_chunks': 59, 'Header 1': 'Chapter 4: Chunking Strategies', 'Chunk_id': 'header_11', 'strategy': 'header'}, page_content='Chapter 4: Chunking Strategies-Chunking is the most impactful decision in your RAG pipeline. This chapter covers all four major strategies with\ncode, tradeoffs, and when to use each.'),
 Document(metadata={'Header 1': 'Appendix F: RAG Interview Questions & Model Answers', 'Header 2': 'F.1 Conceptual Questions', 'strategy': 'header', 'source': 'langchain_rag_technical_docs_clean.pdf', 'Chunk_id': 'header_56', 'total_chunks': 59, 'header_id': 'langchain_rag_technical_docs_clean.pdf-Appendix F: RAG Interview Questions & Model Answers-F.1 Conceptual Questions-56', 'hash': '98b9798a4bd1777c34e4c9a5bfdfecdb'}, page_content="F.1 Conceptual Questions-Q: What is t

In [ ]:
state['sections']

['langchain_rag_technical_docs_clean.pdf-Chapter 4: Chunking Strategies-4.1 Fixed-Size Chunking-12',
 'langchain_rag_technical_docs_clean.pdf-Chapter 6: Vector Stores-6.3 Deduplication with PDF Hashing-22',
 'langchain_rag_technical_docs_clean.pdf-Chapter 10: RAG Evaluation with RAGAS-10.2 Strategy Comparison Evaluation-31',
 'langchain_rag_technical_docs_clean.pdf-Chapter 10: RAG Evaluation with RAGAS-10.1 The Four Core RAG Metrics-30',
 'langchain_rag_technical_docs_clean.pdf-Appendix F: RAG Interview Questions & Model Answers-F.3 Debugging Scenarios-58']

In [ ]:
state['query']

'can you explain different methods of chunking strategies with sample code?'

In [ ]:
state['retry_count']

0

In [ ]:
state['reranked_chunks']

[Document(metadata={'Chunk_id': 'header_7', 'total_chunks': 59, 'source': 'uploaded_doc.pdf', 'Header 1': 'Chapter 2: Python Environment Setup', 'Header 2': '2.4 Project Structure', 'hash': '52c51f407056cbe334d3e5a05bd0ca95', 'header_id': 'uploaded_doc.pdf-Chapter 2: Python Environment Setup-2.4 Project Structure-7', 'strategy': 'header', 'relevance_score': 0.984516}, page_content='2.4 Project Structure-rag-chunking-strategies/\n■■■ src/\n■   ■■■ chunkers/\n■   ■   ■■■ __init__.py\n■   ■   ■■■ header_chunker.py\n■   ■   ■■■ semantic_chunker.py\n■   ■   ■■■ parent_child_chunker.py\n■   ■■■ retriever.py\n■   ■■■ embeddings.py\n■   ■■■ pipeline.py\n■■■ app/\n■   ■■■ streamlit_app.py\n■■■ tests/\n■   ■■■ test_chunkers.py\n■■■ data/\n■   ■■■ sample_docs/\n■■■ .env\n■■■ .env.example\n■■■ requirements.txt\n■■■ README.md'),
 Document(metadata={'hash': '9debbf556a3c412aca436dd65cd56def', 'source': 'uploaded_doc.pdf', 'header_id': 'uploaded_doc.pdf-Chapter 4: Chunking Strategies-MISSING-11', 'to

In [ ]:
state['answer']

'## Introduction to Chunking Strategies\nChunking is a crucial decision in the RAG pipeline, and there are several strategies to choose from. This answer will cover the different methods of chunking strategies with sample code.\n\n## 1. Fixed-Size Chunking\nFixed-size chunking is the simplest approach, where text is split every N characters with an overlap window. This method is fast but semantically unaware.\n\n```python\nfrom langchain.text_splitter import RecursiveCharacterTextSplitter\nsplitter = RecursiveCharacterTextSplitter(\n    chunk_size=500,         # max characters per chunk\n    chunk_overlap=50,       # overlap to preserve context at boundaries\n    length_function=len,\n    separators=["\\n\\n", "\\n", ". ", " ", ""]\n)\nchunks = splitter.split_documents(documents)\nprint(f"Created {len(chunks)} chunks")\nprint(f"Average chunk size: {sum(len(c.page_content) for c in chunks)/len(chunks):.0f} chars")\n```\n\n## 2. Header Chunking\nHeader chunking is another strategy that c